# Lab 5 — Part 3: Billing retry and commit the pre-run estimate

**ผลกระทบของ cell นี้: อ่าน billing และสร้าง Git commit ในเครื่อง เฉพาะ `lab5/reports/lab5-estimate.md` ไม่มี push และไม่เริ่ม compute/training**

อ่านร่าง [lab5-estimate.md](../reports/lab5-estimate.md) ก่อนรัน:
- Part A subtotal **17.3568 THB** ใช้ DS2 v2 Linux on-demand 5.6784 THB/hour
- Proposed envelope **44.8282 THB** รวมเงินสำรองที่เป็นข้อเสนอ ไม่ใช่ราคา/ค่าใช้จริงที่ยืนยันแล้ว
- Compute assumption: training 0.5 h + pipeline/startup/idle 0.5 h + staging 1 h
- ไม่ถือว่าบริการ provider ที่สอง/LLM พร้อมใช้ และยังไม่ยืนยันงบรวมทั้งเทอม
- Quota low-priority รวมเป็น 0; ข้อ discounted compute ยังไม่ผ่าน ห้ามอ้างว่า dedicated ทำให้ครบข้อกำหนดนี้

ใช้ WSL/kernel `itcs355-lab3` เดิม ไม่ต้องรัน Parts 1–2 ใหม่

cell ตรวจว่าไม่มีไฟล์อื่น staged และ estimate ยังตรง SHA-256 จาก Part 2 ก่อนทำงาน
ถ้ามี staged files อยู่ก่อนแล้วจะหยุดโดยไม่แก้ staging ให้ส่ง output มา
อ่าน billing โดยเคารพ Retry-After ไม่ retry ไม่จำกัด และใช้ผล billing สำเร็จที่อายุน้อยกว่า 24h ซ้ำ
หาก server ขอรอเกิน 60s จะเก็บ pending แล้วหยุดเรียก billing ต่อ
**แม้ billing ยัง pending ก็ commit estimate ได้ แต่ยังไม่ใช่สิทธิ์ให้เริ่ม cloud ที่คิดเงิน**

รันซ้ำได้: ถ้า estimate ฉบับนี้อยู่ใน HEAD แล้ว จะไม่สร้าง empty commit
ไม่ใช้ git add . และไม่แก้ Git configuration ให้


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
repo = next((base for base in (here, *here.parents)
             if (base / "lab5/scripts/billing_retry.py").is_file()
             and (base / "lab4/README.md").is_file()), None)
if repo is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
lab = repo / "lab5"
relative = "lab5/reports/lab5-estimate.md"
estimate = repo / relative
expected = json.loads((lab / "reports/planning.json").read_text())["estimate"]["sha256"]
digest = hashlib.sha256(estimate.read_bytes()).hexdigest()
if digest != expected:
    raise RuntimeError("Estimate เปลี่ยนจากฉบับที่ตรวจแล้ว หยุดก่อน commit และแจ้งผู้ช่วยครับ")
def git(*args, check=True):
    return subprocess.run(["git", *args], cwd=repo, text=True, capture_output=True, check=check)
staged = git("diff", "--cached", "--name-only").stdout.strip()
if staged:
    raise RuntimeError("มี staged files เดิม หยุดโดยไม่แก้ staging:\n" + staged)
if git("diff", "--name-only", "--diff-filter=U").stdout.strip():
    raise RuntimeError("มี merge conflicts ต้องตรวจให้เรียบร้อยก่อนครับ")
for name in ("user.name", "user.email"):
    if not git("config", "--get", name, check=False).stdout.strip():
        raise RuntimeError("Git ยังไม่มี " + name + " ให้แจ้งผู้ช่วยก่อน ไม่แก้ config อัตโนมัติ")
print("Review confirmed estimate: Part A 17.3568 THB; proposed envelope 44.8282 THB.", flush=True)
print("Retry billing with bounded backoff; no cloud resources will be created.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.billing_retry"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("Billing script failed before commit; ส่ง output ให้ผู้ช่วยครับ")
if hashlib.sha256(estimate.read_bytes()).hexdigest() != expected:
    raise RuntimeError("Estimate changed during billing check; no commit performed")
if git("diff", "--cached", "--name-only").stdout.strip():
    raise RuntimeError("Staging changed during billing check; no commit performed")
old = git("show", "HEAD:" + relative, check=False)
already = old.returncode == 0 and hashlib.sha256(old.stdout.encode()).hexdigest() == expected
if not already:
    git("add", "--", relative)
    names = git("diff", "--cached", "--name-only").stdout.splitlines()
    if names != [relative]:
        raise RuntimeError("Staged allowlist mismatch; stop and inspect before committing")
    print("Commit only: " + relative, flush=True)
    result = git("commit", "-m", "docs: record Lab 5 estimate before managed execution", "--only", "--", relative)
    print(result.stdout, flush=True)
else:
    print("Estimate is already committed in HEAD; no new commit needed.", flush=True)
commit = git("rev-parse", "HEAD").stdout.strip()
committed = git("show", commit + ":" + relative).stdout
if hashlib.sha256(committed.encode()).hexdigest() != expected:
    raise RuntimeError("Committed estimate verification failed; do not submit a pipeline")
billing = json.loads((lab / "reports/billing-review.json").read_text())
record = {
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "estimate_commit": commit,
    "estimate_sha256": expected,
    "estimate_path": relative,
    "billing_status": {key: billing.get(key, {}).get("status") for key in ("group_billing", "lab5_billing")},
    "resources_started": False,
    "note": "Estimate committed; budget headroom and discounted-compute requirement remain unresolved.",
}
record_path = lab / "reports/estimate-commit.json"
if record_path.exists():
    history = lab / "reports/estimate-commit-history"
    history.mkdir(exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    (history / (stamp + ".json")).write_bytes(record_path.read_bytes())
record_path.write_text(json.dumps(record, indent=2) + "\n")
print("\n--- SEND THIS SUMMARY ---", flush=True)
print(json.dumps(record, indent=2), flush=True)
print("หยุดตรงนี้และส่ง summary ครับ ยังไม่เริ่ม managed pipeline", flush=True)


Review confirmed estimate: Part A 17.3568 THB; proposed envelope 44.8282 THB.
Retry billing with bounded backoff; no cloud resources will be created.
[13:30:41] Read group_billing
[13:30:43] Billing HTTP 429: wait 4s before attempt 2/3
[13:30:50] Read lab5_billing
{
  "group_billing": {
    "status": "ok",
    "posted_totals_by_currency": {
      "USD": 7.896070275116232
    }
  },
  "lab5_billing": {
    "status": "ok",
    "posted_totals_by_currency": {}
  }
}
Full-term THB budget and final Lab 5 actual cost remain unverified. No resources were started.
Commit only: lab5/reports/lab5-estimate.md
[main 83c400d] docs: record Lab 5 estimate before managed execution
 1 file changed, 50 insertions(+)
 create mode 100644 lab5/reports/lab5-estimate.md


--- SEND THIS SUMMARY ---
{
  "recorded_at": "2026-10-23T06:31:01.791611+00:00",
  "estimate_commit": "83c400d616b3323c292d9c028fd80cfd9e1200ba",
  "estimate_sha256": "65e0795089f3748c99a08c9abd44eff94d2558939e24ff7c02f648de7247350a",
  "est

## หลังรัน

ส่ง `SEND THIS SUMMARY` พร้อม billing statuses กลับมา
Git commit จะมี estimate เพียงไฟล์เดียว ไฟล์ source/notebook/reports อื่นยังไม่ได้ commit
ไม่มีการ push ไป GitHub และไม่มี compute/training/endpoint ใหม่

HTTP 429 หมายถึง billing ถูก throttle; หากยังเป็นอยู่จะใช้ Portal Cost Analysis/CSV เป็นทางเลือก
ไม่รายงาน actual=0 และไม่อ้างว่าผ่านงบ <800 THB จากเครดิตที่ยังเหลือ
